# Laboratorium 3 — metoda gradientów sprzężonych

**Metody Optymalizacji · 2026**

Uzupełnij `...` w funkcjach `gd`, `cg` i `normal_eq_solve`, wykonując zadania po kolei. Pozostały kod, w tym testy, generowanie danych i wykresy, jest gotowy. W miejscach na odpowiedzi zapisz uzasadnienia i krótkie wnioski.

In [ ]:
%pip install pydicom nibabel requests

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from time import perf_counter

## 1. Wyprowadzenie metody gradientów sprzężonych (CG)

### Punkt wyjścia

Problem optymalizacji: $f(x)=\frac12x^\top Ax-b^\top x$, gdzie $A=A^\top>0$

Przypomnienie z Lab 1-2:
- Wiemy, że $\nabla f(x)=Ax-b,\qquad Ax^\star=b.$
- Residuum definiujemy jako $r_k=b-Ax_k$.
- Chcemy poruszyć się w tym kierunku z krokiem $\alpha_k$: 
$$
x_{k+1} = x_k + \alpha_k p_k \\
b - A x_{k+1} = b - A x_k - \alpha_k A p_k  \\
r_{k+1} = r_k - \alpha_k A p_k
$$
- Dla dokładnej optymalizacji $\varphi(\alpha) = f(x_k + \alpha p_k)$ w kierunku $p_k$ z punktu $x_k$ mamy (Zadanie 3 Lab 2)
$$
\varphi'(\alpha) = -r_k^T p_k + \alpha\, p_k^T A p_k = 0 \;\Longrightarrow\; \alpha_k = \frac{r_k^T p_k}{p_k^T A p_k}.
$$
- Zatem mamy, że $\varphi'(\alpha_k) = -r_{k+1}^T p_k = 0$, ponieważ $\alpha_k$ jest minimum. 
- Pokazaliśmy także, że dla kierunku spadku dokładny krok $\alpha_k$ jest dodatni i spełnia warunki Wolfe’a przy $0<c_1\le\frac12$ oraz $c_1<c_2<1$.

### Główna idea

W metodzie gradientów sprzężonych chcemy, aby kierunek $p_{k+1}$ był A-sprzężony do wszystkich wcześniejszych kierunków $p_i$ to znaczy:
$$
p_{k+1}^T A p_i = 0 \qquad \text{dla} \qquad i=0,\cdots,k
$$

W metodzie spadku z gradientem nasz kierunek miał postać $p_{k+1} = r_{k+1}$, w CG chcemy wprowadzić liniową poprawkę, aby zapewnić warunek A-sprzężoności.

Niech $p_{k+1} = r_{k+1} + \beta_k p_k$. Wyprowadź wzór na $\beta_k$ tak, aby $p_{k+1}^T A p_{k} = 0$.

**Odpowiedź:** …

### Dowód indukcyjny

Otrzymaliśmy równanie na $\beta_k$, które gwarantuje nam, że $p_{k+1}$ będzie $A$ sprzężony do $p_k$. Teraz indukcyjnie można pokazać, że tak utworzony $p_{k+1}$ jest także $A$ sprzężony do wszystkich $p_i$ dla $i < k$.

**Zadanie dodatkowe** Pokaż indukcyjnie, że dla każdej pary $(i, j)$ takiej, że $i < j$ (indukcja po $j$):
$$\text{(a)}\;\; r_j^T p_i = 0, \qquad \text{(b)}\;\; p_j^T A p_i = 0, \qquad \text{(c)}\;\; r_j^T r_i = 0.$$

1. Zacznij od bazy indukcyjnej $(i, j)$ = $(0, 1)$
2. Załóż, że warunki (a)-(c) są spełnione dla każdej pary $(i, j)$ $i < j \leq k$.
3. Pokaż, że warunki (a)-(c) są spełnione dla par $(i, j = k+1)$ dla $i \leq k$. 

Wskazówki: 
- Zacznij od warunku (a), potem (c), potem (b).
- Skorzystaj ze wzorów:

$$ \varphi'(\alpha_k) = -r_{k+1}^T p_k = 0 \tag{*} $$
$$ p_{k+1} = r_{k+1} + \beta_k p_k \tag{**} $$
$$ r_{k+1} = r_k - \alpha_k A p_k \tag{***} $$

**Odpowiedź:** …

### Uproszczenie wzoru na $\beta_k$

Wykorzystując stwierdzenia (a)-(c) z dowodu pokaż, że
$$
\alpha_k = \frac{r_k^T r_k}{p_k^T A p_k} \quad \text{oraz} \quad
\beta_k = \frac{r_{k+1}^T r_{k+1}}{r_{k}^T r_{k}}.
$$

**Odpowiedź:** …

## 2. Implementacja algorytmu metody gradientów sprzężonych CG

### Pseudokod algorytmu GD

**Dane:** $A=A^\top>0$, $b$, punkt startowy $x_0$, tolerancja $\varepsilon>0$.

1. $r_0 = b - Ax_0$, $\;p_0 = r_0$
2. Dla $k = 0, 1, 2, \dots$:
   1. jeśli $\|r_k\| \le \varepsilon$ (np. $\varepsilon\|b\|$), **zakończ** i zwróć $x_k$
   2. $\alpha_k = \dfrac{r_k^Tr_k}{p_k^TAp_k}$
   3. $x_{k+1} = x_k + \alpha_k p_k$
   4. $p_{k+1} = r_{k+1} = r_k - \alpha_k Ap_k$

Uzupełnij cztery miejsca `...` w poniższej wersji kodu GD z Lab 2, tak aby pasowała do powyższego pseudokodu.

In [ ]:
def gd(A, b, x0, eps=1e-6, max_iter=20_000,
       keep_history=True, time_limit=np.inf):
    start = perf_counter()

    x = np.array(x0, dtype=float)
    r = b - A @ x  # residuum w punkcie początkowym
    p = r

    X = [x.copy()] if keep_history else []
    R = [np.linalg.norm(r)]
    T = [0.]

    status = "limit iteracji"
    for k in range(max_iter):
        if not np.isfinite(R[-1]):
            status = "nieskończone residuum"
            break
        if R[-1] <= eps:
            break
        if perf_counter() - start >= time_limit:
            status = "limit czasu"
            break
        
        Ap = A @ p
        alpha = (r @ r) / (p @ Ap)  # dokładny krok

        x = x + alpha * p  # nowy punkt
        r = r - alpha * Ap  # aktualizacja residuum
        p = r 

        if keep_history:
            X.append(x.copy())
        R.append(np.linalg.norm(r))
        T.append(perf_counter() - start)
    if R[-1] <= eps:
        status = "OK"
    return {"x": x, "X": np.array(X), "R": np.array(R),
            "T": np.array(T), "status": status}

In [ ]:
A = np.eye(2)
b = np.array([1., 2.])
result = gd(A, b, np.zeros(2))
assert np.allclose(result["x"], b)
assert len(result["R"]) - 1 == 1
assert result["status"] == "OK"
print("Liczba kroków:", len(result["R"]) - 1)

### Pseudokod algorytmu CG

**Dane:** $A=A^\top>0$, $b$, punkt startowy $x_0$, tolerancja $\varepsilon>0$.

1. $r_0 = b - Ax_0$, $\;p_0 = r_0$
2. Dla $k = 0, 1, 2, \dots$:
   1. jeśli $\|r_k\| \le \varepsilon$ (np. $\varepsilon\|b\|$), **zakończ** i zwróć $x_k$
   2. $\alpha_k = \dfrac{r_k^Tr_k}{p_k^TAp_k}$
   3. $x_{k+1} = x_k + \alpha_k p_k$
   4. $r_{k+1} = r_k - \alpha_k Ap_k$
   5. $\beta_k = \dfrac{r_{k+1}^Tr_{k+1}}{r_k^Tr_k}$
   6. $p_{k+1} = r_{k+1} + \beta_k p_k$

**Uwaga:** Iloczyn macierz-wektor $Ap_k$ może być przeliczony raz i wykorzystany dwa razy (w punkcie 2 oraz 4)


Wykorzystując uzupełnioną funkcję `gd`, uzupełnij pięć miejsc `...` w funkcji `cg`. Zachowujemy te same parametry i format wyniku; nowym elementem jest poprawka kierunku.

In [ ]:
def cg(A, b, x0, eps=1e-6, max_iter=20_000,
       keep_history=True, time_limit=np.inf):
    start = perf_counter()

    x = np.array(x0, dtype=float)
    r = b - (A @ x0)
    p = r

    X = [x.copy()] if keep_history else []
    R = [np.linalg.norm(r)]
    T = [0.]

    status = "limit iteracji"
    for k in range(max_iter):
        if not np.isfinite(R[-1]):
            status = "nieskończone residuum"
            break
        if R[-1] <= eps:
            break
        if perf_counter() - start >= time_limit:
            status = "limit czasu"
            break
        
        Ap = A @ p
        alpha = (r @ r) / (p @ Ap)  # dokładny krok

        x = x + alpha * p  # nowy punkt
        r_next = r - alpha * Ap  # nowe residuum

        beta = (r_next @ r_next) / (r @ r)  # współczynnik poprawki

        p = r_next + beta * p  # nowy kierunek
        r = r_next

        if keep_history:
            X.append(x.copy())
        R.append(np.linalg.norm(r))
        T.append(perf_counter() - start)
    if R[-1] <= eps:
        status = "OK"
    return {"x": x, "X": np.array(X), "R": np.array(R),
            "T": np.array(T), "status": status}

In [ ]:
A = np.eye(2)
b = np.array([1., 2.])
result = cg(A, b, np.zeros(2))
assert np.allclose(result["x"], b)
assert len(result["R"]) - 1 == 1
assert result["status"] == "OK"
print("Macierz jednostkowa — liczba kroków:", len(result["R"]) - 1)

# Ten test odróżnia CG od GD: obie składowe błędu są niezerowe.
A = np.diag([1., 40.])
x_star = np.ones(2)
b = A @ x_star
result = cg(A, b, np.zeros(2), eps=1e-10)
assert np.allclose(result["x"], x_star, rtol=0, atol=1e-10)
assert len(result["R"]) - 1 == 2
assert result["status"] == "OK"
print("Macierz diag(1, 40) — liczba kroków:", len(result["R"]) - 1)


## 3. Porównanie metod

Porównamy GD i CG na funkcji kwadratowej $f(x)=\frac12x^\top Ax-b^\top x$ w tych samych układach co w Lab 2. W arytmetyce dokładnej CG kończy się po co najwyżej $n$ krokach. Z teorii:
$$\frac{E_k^{GD}}{E_0}\le\left(\frac{\kappa-1}{\kappa+1}\right)^{2k},\qquad
\frac{E_k^{CG}}{E_0}\le 4\left(\frac{\sqrt\kappa-1}{\sqrt\kappa+1}\right)^{2k},\qquad E_k=\|x_k-x^\star\|_A^2 .$$
Do osiągnięcia zadanej dokładności GD potrzebuje więc $O(\kappa)$ iteracji, a CG tylko $O(\sqrt\kappa)$.

In [ ]:
def positive(values):
    values = np.asarray(values)
    return np.where(values > 0, values, np.nan)


def energy(A, x, x_star):
    e = x - x_star
    return e @ A @ e


def make_spd(values, seed=2026):
    rng = np.random.default_rng(seed)
    n = len(values)
    qr_result = np.linalg.qr(rng.normal(size=(n, n)))
    Q = qr_result[0]
    A = (Q * values) @ Q.T
    return (A + A.T) / 2

#### 3.1 Geometria w 2D

$A=\operatorname{diag}(1,\kappa)$, $x^\star=(1,1)$, start $x_0=x^\star+v/\|v\|_2$ z najgorszym dla GD kierunkiem $v=(1,1/\kappa)$ (jak w Lab 2). Porównaj trajektorie GD i CG. Ile kroków wykonuje CG? Wyjaśnij wynik, korzystając ze sprzężoności kierunków.

**Wnioski:** …

In [ ]:
def geometry_cmp(kappa, v, eps=1e-10):
    A = np.diag([1., kappa])
    x_star = np.ones(2)
    b = A @ x_star
    x0 = x_star + v / np.linalg.norm(v)
    fig = plt.figure(figsize=(15, 4))
    outer = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1])
    inner = outer[1].subgridspec(2, 1, hspace=0.15)
    ax_traj = fig.add_subplot(outer[0])
    ax_x0 = fig.add_subplot(inner[0])
    ax_x1 = fig.add_subplot(inner[1], sharex=ax_x0)
    ax_err = fig.add_subplot(outer[2])
    grid = np.meshgrid(np.linspace(-0.2, 2.2, 200), np.linspace(-0.2, 2.2, 200))
    u = grid[0]
    w = grid[1]
    ax_traj.contour(u, w, ((u-1)**2 + kappa*(w-1)**2)/2,
                    levels=np.geomspace(0.001, kappa, 12), colors="grey", alpha=0.4)
    for method_entry in [("GD", gd), ("CG", cg)]:
        name = method_entry[0]
        method = method_entry[1]
        result = method(A, b, x0, eps=eps)
        X = result["X"]
        E = np.array([energy(A, x, x_star) for x in X])
        ax_traj.plot(X[:51, 0], X[:51, 1], ".-", label=name, markersize=4)
        ax_x0.plot(X[:51, 0], ".-", label=name, markersize=3)
        ax_x1.plot(X[:51, 1], ".-", label=name, markersize=3)
        ax_err.semilogy(positive(E / E[0]), ".-", label=name, markersize=3)
        print(f"κ = {kappa:3g} | {name} | kroki: {len(E)-1:4d} | "
              f"residuum: {result['R'][-1]:.2e} | {result['status']}")
    q = (kappa - 1) / (kappa + 1)
    ax_err.semilogy(q**(2*np.arange(60)), "k--", label="oszacowanie GD: q^(2k)")
    ax_traj.plot(1, 1, "k*", markersize=10)
    ax_traj.set(xlabel="x₀", ylabel="x₁", title=f"κ = {kappa:g}, pierwsze 50 kroków")
    ax_traj.set_aspect("equal")
    ax_err.set(xlabel="Numer iteracji k", ylabel="Eₖ / E₀", xlim=(-1, 60), ylim=(1e-14, 2),
               title="Relatywna energia w kolejnych iteracjach")
    ax_x0.axhline(1, color="k", ls="--", lw=0.8)
    ax_x1.axhline(1, color="k", ls="--", lw=0.8)
    ax_x0.set(ylabel="x₀", title="Zmienne w kolejnych iteracjach")
    ax_x1.set(xlabel="Numer iteracji k", ylabel="x₁")
    plt.setp(ax_x0.get_xticklabels(), visible=False)
    for ax in [ax_traj, ax_x0, ax_x1, ax_err]:
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()

In [ ]:
for kappa in [10., 40.]:
    geometry_cmp(kappa, np.array([1., 1/kappa]))

#### 3.2 Zbieżność w wyższym wymiarze

Macierze jak w zadaniu 4 z Lab 2: $A=Q\Lambda Q^\top$ z $\Lambda=\operatorname{diag}(\mathrm{linspace}(1,\kappa,n))$. Wykresy pokazują $E_k/E_0$ oraz $\|r_k\|/\|r_0\|$ dla obu metod wraz z oszacowaniami z teorii.

In [ ]:
def convergence_cmp(n, kappa, eps=1e-8, max_iter=200_000):
    A = make_spd(np.linspace(1., kappa, n))
    x_star = np.ones(n)
    b = A @ x_star
    x0 = np.zeros(n)
    plot = plt.subplots(1, 2, figsize=(11, 4))
    fig = plot[0]
    ax = plot[1]
    sq = np.sqrt(kappa)
    q_gd = (kappa - 1) / (kappa + 1)
    q_cg = (sq - 1) / (sq + 1)
    steps = {}
    for method_entry in [("GD", gd), ("CG", cg)]:
        name = method_entry[0]
        method = method_entry[1]
        result = method(A, b, x0, eps=eps, max_iter=max_iter)
        E = np.array([energy(A, x, x_star) for x in result["X"]])
        steps[name] = len(E) - 1
        ax[0].semilogy(positive(E / E[0]), label=name)
        ax[1].semilogy(positive(result["R"] / result["R"][0]), label=name)
        print(f"n = {n}, κ = {kappa:g} | {name} | kroki: {steps[name]:6d} | "
              f"residuum: {result['R'][-1]:.2e} | {result['status']}")
    k = np.arange(steps["GD"] + 1)
    ax[0].semilogy(q_gd**(2*k), "k--", label="oszacowanie GD: q^(2k), q=(κ-1)/(κ+1)")
    ax[0].semilogy(4*q_cg**(2*k), "k:", label="oszacowanie CG: 4·q_CG^(2k), q_CG=(√κ-1)/(√κ+1)")
    ax[0].set(xlabel="Numer iteracji k", ylabel="Eₖ / E₀", ylim=(1e-16, 10),
              title=f"Energia błędu, n = {n}, κ = {kappa:g}")
    ax[1].set(xlabel="Numer iteracji k", ylabel="‖rₖ‖ / ‖r₀‖",
              title="Relatywna norma residuum")
    for a in ax:
        a.legend(fontsize=8)
        a.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()
    return steps

In [ ]:
for kappa in [10., 100., 1000.]:
    convergence_cmp(1000, kappa)

#### 3.3 Liczba iteracji w zależności od $\kappa$

Dla rosnącego $\kappa$ (stałe $n$) liczymy kroki potrzebne do $\|r_k\|\le\varepsilon\|b\|$. Na wykresie log-log nachylenie $1$ odpowiada $O(\kappa)$, a $1/2$ odpowiada $O(\sqrt\kappa)$.

In [ ]:
def scaling_cmp(n, kappas, eps=1e-8, max_iter=500_000, repeats=5):
    rows = {"GD": [], "CG": []}
    times = {"GD": [], "CG": []}
    spread = {"GD": [], "CG": []}  # (min, max) czasu z powtórzeń
    for kappa in kappas:
        A = make_spd(np.linspace(1., kappa, n))
        b = A @ np.ones(n)
        for method_entry in [("GD", gd), ("CG", cg)]:
            name = method_entry[0]
            method = method_entry[1]
            runs = [method(A, b, np.zeros(n), eps=eps*np.linalg.norm(b),
                           max_iter=max_iter, keep_history=False)
                    for repetition in range(repeats)]
            rows[name].append(len(runs[0]["R"]) - 1)
            t = [run["T"][-1] for run in runs]
            times[name].append(np.median(t))
            spread[name].append((min(t), max(t)))
    kappas = np.array(kappas, dtype=float)
    plot = plt.subplots(1, 2, figsize=(11, 4.5))
    fig = plot[0]
    ax = plot[1]
    for plot_entry in [(ax[0], rows, "Liczba iteracji do ‖r‖ ≤ ε‖b‖"),
                       (ax[1], times, f"Czas wykonania [s] (mediana, min–max z {repeats} powtórzeń)")]:
        a = plot_entry[0]
        data = plot_entry[1]
        ylabel = plot_entry[2]
        for method_entry in [("GD", "o-"), ("CG", "s-")]:
            name = method_entry[0]
            marker = method_entry[1]
            if data is times:  # słupki błędów: zakres min–max z powtórzeń
                lo = np.array(data[name]) - np.array(spread[name])[:, 0]
                hi = np.array(spread[name])[:, 1] - np.array(data[name])
                a.errorbar(kappas, data[name], yerr=[lo, hi], fmt=marker,
                           capsize=3, label=name)
            else:
                a.plot(kappas, data[name], marker, label=name)
        a.set(xscale="log", yscale="log")
        c = data["GD"][0] / kappas[0]
        a.loglog(kappas, c*kappas, "k--", lw=0.8, label="~ κ")
        c = data["CG"][0] / np.sqrt(kappas[0])
        a.loglog(kappas, c*np.sqrt(kappas), "k:", lw=0.8, label="~ √κ")
        a.set(xlabel="κ", ylabel=ylabel, title=f"n = {n}, ε = {eps:g}")
        a.legend(fontsize=8)
        a.grid(alpha=0.2, which="both")
    fig.tight_layout()
    plt.show()
    print(f"{'κ':>8} | {'GD kroki':>9} {'GD czas [s]':>12} | {'CG kroki':>9} {'CG czas [s]':>12} | przyspieszenie")
    for i in range(len(kappas)):
        print(f"{kappas[i]:8g} | {rows['GD'][i]:9d} {times['GD'][i]:12.3e} | "
              f"{rows['CG'][i]:9d} {times['CG'][i]:12.3e} | {times['GD'][i]/times['CG'][i]:.1f}x")
    for data_entry in [("iteracje", rows), ("czas", times)]:
        label = data_entry[0]
        data = data_entry[1]
        for name in data:
            s = np.polyfit(np.log(kappas), np.log(data[name]), 1)[0]
            print(f"{name} ({label}): nachylenie log-log = {s:.2f}")
    return rows, times

In [ ]:
scaling_result = scaling_cmp(1000, [10, 30, 100, 300, 1000, 3000])
rows = scaling_result[0]
times = scaling_result[1]

**Pytania do porównania metod:**

1. Jak zmieniają się liczby iteracji GD i CG wraz ze wzrostem $\kappa$? Porównaj wyniki z oszacowaniami z teorii. Czy oszacowanie z góry musi dobrze przybliżać rzeczywistą liczbę kroków?
2. Czy w obliczeniach zmiennoprzecinkowych należy oczekiwać zakończenia CG po co najwyżej $n$ krokach? Uzasadnij odpowiedź.
3. Porównaj koszt pojedynczej iteracji obu metod. Czy różnice w liczbie kroków przekładają się na podobne różnice w czasie wykonania?

**Wnioski:** …

## 4. Metoda najmniejszych kwadratów — jak zastosować CG?

### 4.1 Od macierzy prostokątnej do znanego zadania kwadratowego

Mamy macierz $A\in\mathbb R^{m\times n}$ i dane $y\in\mathbb R^m$. Szukamy wektora $x\in\mathbb R^n$, dla którego $Ax$ jest możliwie bliskie $y$. Liczba $n$ oznacza liczbę niewiadomych, a $m$ — liczbę pomiarów, czyli równań.

Może być zarówno $m>n$, jak i $m<n$. Więcej równań niż niewiadomych nie gwarantuje rozwiązania równania $Ax=y$ ani niezależności kolumn $A$. Dlatego rozpatrujemy **zadanie najmniejszych kwadratów**:
$$
\min_x f(x),\qquad f(x)=\frac12\|Ax-y\|_2^2.
$$
Macierz $A$ może być prostokątna, ale funkcja $f$ nadal jest kwadratowa. Rozwijając kwadrat, otrzymujemy
$$
f(x)=\frac12x^\top A^\top Ax-x^\top A^\top y+\frac12y^\top y.
$$
Ostatni składnik nie zależy od $x$. Oznaczmy
$$
B=A^\top A\in\mathbb R^{n\times n},\qquad b=A^\top y\in\mathbb R^n.
$$
Otrzymaliśmy znaną postać $\frac12x^\top Bx-b^\top x$, z dokładnością do stałej. Gradient wynosi $\nabla f(x)=Bx-b$, a warunek optymalności to **równania normalne**:
$$
Bx=b,\qquad\text{czyli}\qquad A^\top Ax=A^\top y.
$$
To do tego układu chcemy zastosować CG. Pozostaje sprawdzić, co wiemy o macierzy $B$.

### 4.2 Co zmienia dodatnia półokreśloność?

Macierz $B=A^\top A$ jest symetryczna i dla każdego $v$ spełnia
$$
v^\top Bv=\|Av\|_2^2\ge0.
$$
Jest więc dodatnio półokreślona. Jeśli kolumny $A$ są liniowo niezależne, to $Av\ne0$ dla $v\ne0$, więc $B$ jest dodatnio określona i możemy od razu użyć CG.

**A jeśli kolumny są zależne?** Czyli np. gdy $m < n$? Istnieje wtedy niezerowe $z$ takie, że $Az=0$. Wektory $x$ i $x+z$ dają identyczne pomiary:
$$
A(x+z)=Ax.
$$
Pomiary nie pozwalają ich rozróżnić. Minimum zadania najmniejszych kwadratów istnieje, ale nie jest jednoznaczne. Taka sytuacja może wystąpić również przy $m\ge n$.

**Dlaczego CG nadal działa?** Zaczynamy od $x_0=0$ i rozważamy przestrzeń
$$
V=\operatorname{im}(A^\top)=(\ker A)^\perp.
$$
Wektor $b=A^\top y$ należy do $V$, a $Bv=A^\top(Av)$ również należy do $V$. Ze wzorów CG wynika więc, że punkty, residua układu $Bx=b$ i kierunki pozostają w $V$.

Dla niezerowego $v\in V$ mamy $Av\ne0$: wektor należący jednocześnie do $\ker A$ i do $(\ker A)^\perp$ musi być zerowy. Zatem
$$
v^\top Bv=\|Av\|_2^2>0\qquad\text{dla }v\in V\setminus\{0\}.
$$
**W przestrzeni, w której wykonujemy iteracje, macierz jest dodatnio określona.** Możemy tam zastosować teorię CG. W arytmetyce dokładnej metoda znajduje rozwiązanie po co najwyżej $\dim V=\operatorname{rank}A$ krokach. Jeśli $b=0$, rozwiązaniem jest już $x_0=0$.

Nie musimy wyznaczać bazy $V$ ani zmieniać wzorów CG — wystarczy start od zera i zatrzymanie po znalezieniu rozwiązania. Otrzymany minimizer $x^\star$ należy do $V$. Każdy inny ma postać $x^\star+z$, gdzie $z\in\ker A$, więc
$$
\|x^\star+z\|_2^2=\|x^\star\|_2^2+\|z\|_2^2.
$$
CG wybiera zatem **minimizer o najmniejszej normie**. Nie oznacza to jeszcze, że w zastosowaniu będzie to prawdziwy obraz: pomiary nie zawierają informacji o składowej należącej do $\ker A$.

**Pytanie:** Sprawdź na wzorach CG, dlaczego z $x_0=0$ i $b\in V$ wynika, że wszystkie iteracje pozostają w $V$.

### 4.3 Nie musimy tworzyć macierzy $A^\top A$

W CG potrzebujemy iloczynu macierzy układu przez kierunek $p$. Możemy go obliczyć w dwóch krokach:
$$
q=Ap,\qquad Bp=A^\top q=A^\top(Ap).
$$
Nie trzeba przechowywać macierzy $B=A^\top A$, która może być znacznie gęstsza niż $A$. Wystarczą dwie funkcje: jedna obliczająca $Av$, druga obliczająca $A^\top w$.

W poniższym kodzie nazywają się one `A` i `AH`. **Na razie wszystkie dane są rzeczywiste, więc `AH(w)` oznacza po prostu $A^\top w$.** Takie zapisanie CG dla zadania najmniejszych kwadratów nazywamy CGLS (Conjugate Gradient Least Squares).

### 4.4 Wzory do implementacji i zatrzymanie metody

Trzeba odróżnić dwie różnice:
$$
r_k=y-Ax_k\in\mathbb R^m,\qquad
s_k=A^\top r_k=b-Bx_k\in\mathbb R^n.
$$
- $r_k$ jest **residuum danych**: mówi, jak dobrze $Ax_k$ pasuje do $y$.
- $s_k$ jest **residuum równań normalnych**: mamy $s_k = -\nabla f(x_k)$ i to ono odgrywa rolę residuum w CG.

Dla $x_0=0$ przyjmujemy $r_0=y$, $s_0=A^\top y$ i $p_0=s_0$. W jednej iteracji obliczamy
$$
q_k=Ap_k,\qquad
\alpha_k=\frac{\|s_k\|_2^2}{\|q_k\|_2^2},\qquad
x_{k+1}=x_k+\alpha_kp_k.
$$
Mianownik to znane z CG $p_k^\top Bp_k=\|Ap_k\|_2^2$. Następnie aktualizujemy
$$
r_{k+1}=r_k-\alpha_kq_k,\qquad s_{k+1}=A^\top r_{k+1}.
$$
W GD wybieramy $p_{k+1}=s_{k+1}$. W CG dodajemy poprawkę:
$$
\beta_k=\frac{\|s_{k+1}\|_2^2}{\|s_k\|_2^2},\qquad
p_{k+1}=s_{k+1}+\beta_kp_k.
$$
W obu wariantach wykonujemy jedno mnożenie przez $A$ i jedno przez $A^\top$ na iterację.

**Kiedy zakończyć?** Jeśli $s_k=0$, znaleźliśmy minimum. W obliczeniach możemy użyć tolerancji $\|s_k\|_2\le\varepsilon$. Nie wymagamy $r_k=0$: najlepsze dopasowanie może pozostawiać niezerowy błąd w równaniach $Ax=y$. Warunek zatrzymania sprawdzamy przed dzieleniem we wzorach algorytmu, także w punkcie początkowym.

Uzupełnij osiem miejsc `...` w funkcji `normal_eq_solve`, korzystając z powyższych wzorów. Argumenty `A` i `AH` są funkcjami realizującymi mnożenie. W kodzie `np.vdot(v, v).real` oblicza $\|v\|_2^2$, a `gamma` przechowuje $\|s_k\|_2^2$.

Parametr `n_iter` określa maksymalną liczbę kroków, a `eps` — tolerancję bezwzględną dla $\|s_k\|_2$ (domyślnie $10^{-10}$). Funkcja sprawdza zatrzymanie także przed pierwszą iteracją i zabezpiecza dzielenie przez zerowy mianownik. Zwraca liczbę wykonanych kroków (`iterations`) i przyczynę zakończenia (`status`). Zapisuje też ostatni punkt, aby wykresy działały po wcześniejszym zatrzymaniu. Uzupełnij w implementacji zarówno wzory iteracji, jak i sprawdzenie warunku stopu.

W teście poniżej nazwa `B` w kodzie oznacza prostokątną macierz danych — macierz nazywaną $A$ w naszym wyprowadzeniu. Porównujemy wynik z rozwiązaniem uzyskanym przez `np.linalg.lstsq`.

In [ ]:
def normal_eq_solve(A, AH, y, method="cg", n_iter=50, x_ref=None,
                    save_at=(), eps=1e-10):
    """GD lub CG dla AᴴA x = Aᴴ y, bez budowania AᴴA. Start: x0 = 0.
    Stop: norma residuum równań normalnych <= eps lub limit n_iter."""
    if method not in ("gd", "cg"):
        raise ValueError("Nieznana metoda")
    if eps < 0 or not np.isfinite(eps):
        raise ValueError("Tolerancja musi być skończona i nieujemna")
    if n_iter < 0:
        raise ValueError("Limit iteracji musi być nieujemny")
    start = perf_counter()

    r = np.array(y, copy=True)             # residuum danych r = y - A x
    s = AH(r)                            # residuum równań normalnych s = Aᴴ r
    x = np.zeros_like(s)
    p = s.copy()
    gamma = np.vdot(s, s).real             # Kwadrat normy residuum

    nrm = np.linalg.norm
    err = [nrm(x - x_ref) / nrm(x_ref)] if x_ref is not None else []
    res = [nrm(r)]
    nres = [nrm(s)]
    T = [0.]
    snaps = {}
    if 0 in save_at:
        snaps[0] = (x.copy(), r.copy())
    iterations = 0
    status = "limit iteracji"

    # Sprawdzamy stop przed pierwszym krokiem i po każdym kolejnym.
    for k in range(n_iter + 1):
        if not np.isfinite(gamma) or not np.isfinite(nres[-1]):
            status = "nieskończone residuum"
            break
        if nres[-1] <= eps:  # warunek zatrzymania
            status = "OK"
            break
        if k == n_iter:
            break

        q = A(p)                         # jedyny A(.) w iteracji
        denominator = np.vdot(q, q).real
        if not np.isfinite(denominator):
            status = "nieskończony mianownik"
            break
        if denominator <= 0:
            status = "zerowy mianownik"
            break
        alpha = gamma / denominator  # dokładny krok

        x = x + alpha * p  # nowy punkt
        r = r - alpha * q  # nowe residuum danych
        s = AH(r)                       # jedyny AH(.) w iteracji
        gamma_new = np.vdot(s, s).real
        if method == "cg":
            p = s + (gamma_new / gamma) * p  # nowy kierunek CG
        else:
            p = s
        gamma = gamma_new
        iterations = k + 1

        res.append(nrm(r))
        nres.append(nrm(s))
        T.append(perf_counter() - start)
        if x_ref is not None:
            err.append(nrm(x - x_ref) / nrm(x_ref))
        if iterations in save_at:
            snaps[iterations] = (x.copy(), r.copy())

    # Ostatni punkt jest dostępny również po wcześniejszym zatrzymaniu.
    snaps[iterations] = (x.copy(), r.copy())
    return {"x": x, "err": np.array(err), "res": np.array(res),
            "nres": np.array(nres), "T": np.array(T), "snaps": snaps,
            "iterations": iterations, "status": status}


In [ ]:
# Test na małym, gęstym problemie nadokreślonym
rng = np.random.default_rng(0)
m = 300
n = 100
B = rng.normal(size=(m, n)) * np.logspace(0, -1, n)   # kolumny o malejących skalach
x_star = rng.normal(size=n)
yB = B @ x_star
sv = np.linalg.svd(B, compute_uv=False)
print(f"κ(B) = {sv[0]/sv[-1]:.1f},  κ(BᵀB) = {np.linalg.cond(B.T @ B):.1f}  (κ(B)² = {(sv[0]/sv[-1])**2:.1f})")

out = normal_eq_solve(lambda v: B @ v, lambda v: B.T @ v, yB, "cg", n_iter=150, x_ref=x_star)
least_squares_result = np.linalg.lstsq(B, yB, rcond=None)
x_ls = least_squares_result[0]
print("‖x_CG − x_lstsq‖/‖x_lstsq‖ =", np.linalg.norm(out["x"] - x_ls) / np.linalg.norm(x_ls))
assert np.allclose(out["x"], x_ls, rtol=1e-5, atol=1e-8)

## 5. Rekonstrukcja obrazów: tomografia komputerowa i rezonans magnetyczny

Teraz wykorzystamy ten sam algorytm do odtworzenia obrazu z pomiarów. W obu przykładach:

- $x_{\mathrm{true}}$ oznacza obraz wzorcowy, a $x$ — szukany obraz;
- $A$ opisuje, jak z obrazu powstają pomiary;
- $y=Ax_{\mathrm{true}}+\eta$ jest wektorem pomiarów, do których dodaliśmy szum $\eta$.

Obraz wzorcowy pobieramy z gotowego zbioru, ale **pomiary generujemy sami**. Algorytm korzysta tylko z $A$ i $y$. Znany wzorzec służy nam do sprawdzenia, jak dobry obraz otrzymaliśmy.

Rozwiązujemy zadanie
$$
\min_x f(x),\qquad f(x)=\frac12\|Ax-y\|_2^2,
$$
zaczynając od $x_0=0$. Najpierw rozpatrzymy rzeczywistą macierz pomiarów w CT. Wtedy gradient ma postać
$$
\nabla f(x)=A^\top(Ax-y)=A^\top Ax-A^\top y.
$$
Aby go obliczyć, najpierw wyznaczamy przewidywane pomiary $Ax$, odejmujemy dane $y$, a następnie mnożymy różnicę przez $A^\top$. Warunek $\nabla f(x)=0$ prowadzi do równań normalnych
$$
A^\top Ax=A^\top y,
$$
które rozwiązujemy metodą CG z sekcji 4. Używane w kodzie $s_k=A^\top(y-Ax_k)$ jest więc przeciwieństwem gradientu: $s_k=-\nabla f(x_k)$.

Dopiero w przykładzie MRI wprowadzimy liczby zespolone i odpowiedni wzór na gradient.

Poniższe funkcje rysujące są gotowe. `show_reconstruction` pokazuje obraz wzorcowy, pomiary i wyniki po wybranych liczbach iteracji. `plot_curves` pokazuje trzy różne miary:

| Miara | Na jakie pytanie odpowiada? |
|---|---|
| Błąd obrazu $\|x_k-x_{\mathrm{true}}\|_2$ | Czy odtworzony obraz jest bliski wzorcowi? |
| Residuum danych $\|y-Ax_k\|_2$ | Czy obraz dobrze pasuje do pomiarów? |
| Residuum równań normalnych $\|A^\top(y-Ax_k)\|_2$ | Czy gradient funkcji celu jest mały? |

Wykresy pokazują te wielkości po podzieleniu odpowiednio przez $\|x_{\mathrm{true}}\|_2$, $\|y\|_2$ i $\|A^\top y\|_2$. Oznaczenie $A^H$ na wykresie oznacza tutaj $A^\top$; jego znaczenie dla liczb zespolonych wyjaśnimy przy MRI. Zwróć uwagę, czy wszystkie trzy miary poprawiają się jednocześnie.

In [ ]:
def show_reconstruction(x_true, y, runs, ks, show_x, show_y, y_label, y_kw=None, res_kw=None):
    """Wiersze: GD x_k, GD y-A x_k, CG x_k, CG y-A x_k. Pierwsza kolumna: oryginał i dane."""
    y_kw = y_kw or {}
    res_kw = res_kw or y_kw
    ncol = len(ks) + 1
    plot = plt.subplots(4, ncol, figsize=(3.3 * ncol, 11), squeeze=False)
    fig = plot[0]
    axes = plot[1]
    xs = [show_x(x_true)]
    for name in runs:
        run = runs[name]
        for requested_k in ks:
            k = min(requested_k, run["iterations"])
            xs.append(show_x(run["snaps"][k][0]))
    vmax = max(np.max(a) for a in xs)
    names = list(runs)
    for i in range(len(names)):
        name = names[i]
        run = runs[name]
        axes[2*i, 0].imshow(show_x(x_true), cmap="gray", vmin=0, vmax=vmax)
        axes[2*i, 0].set_title("oryginał x")
        axes[2*i+1, 0].imshow(show_y(y), cmap="gray", **y_kw)
        axes[2*i+1, 0].set_title(f"dane y ({y_label})")
        for j in range(len(ks)):
            k = min(ks[j], run["iterations"])
            snapshot = run["snaps"][k]
            xk = snapshot[0]
            rk = snapshot[1]
            suffix = " (koniec)" if k == run["iterations"] else ""
            axes[2*i, j+1].imshow(show_x(xk), cmap="gray", vmin=0, vmax=vmax)
            e = np.linalg.norm(xk - x_true) / np.linalg.norm(x_true)
            axes[2*i, j+1].set_title(f"{name}: x, k = {k}{suffix}  (błąd {e:.2f})")
            axes[2*i+1, j+1].imshow(show_y(rk), cmap="gray", **res_kw)
            axes[2*i+1, j+1].set_title(f"{name}: y − A x_k, k = {k}{suffix}")
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    fig.tight_layout()
    plt.show()


def plot_curves(runs, n_iter, title=""):
    plot = plt.subplots(1, 3, figsize=(15, 4))
    fig = plot[0]
    ax = plot[1]
    for name in runs:
        run = runs[name]
        k = np.arange(len(run["err"]))
        lines = ax[0].semilogy(k, run["err"], label=name)
        line = lines[0]
        kbest = int(np.argmin(run["err"]))
        ax[0].plot(kbest, run["err"][kbest], "o", color=line.get_color())
        res_scale = run["res"][0] if run["res"][0] > 0 else 1.
        nres_scale = run["nres"][0] if run["nres"][0] > 0 else 1.
        ax[1].semilogy(run["res"] / res_scale, label=name)
        ax[2].semilogy(run["nres"] / nres_scale, label=name)
    ax[0].set(xlabel="Numer iteracji k", ylabel="‖xₖ − x_true‖ / ‖x_true‖",
              title="Błąd obrazu (kropka: najlepszy k)")
    ax[1].set(xlabel="Numer iteracji k", ylabel="‖y − Axₖ‖ / ‖y‖",
              title="Residuum danych")
    ax[2].set(xlabel="Numer iteracji k", ylabel="‖Aᴴ(y − Axₖ)‖ / ‖Aᴴy‖",
              title="Residuum równań normalnych")
    for a in ax:
        a.legend(fontsize=8)
        a.grid(alpha=0.2)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    for name in runs:
        run = runs[name]
        e = run["err"]
        print(f"{name}: najlepszy błąd {e.min():.3f} w k = {int(np.argmin(e))}, "
              f"błąd po {run['iterations']} krokach {e[-1]:.3f}; "
              f"{run['status']} (limit: {n_iter})")


### 5.1 Tomografia komputerowa — rzeczywista, rzadka macierz pomiarów

Wyobraźmy sobie promień przechodzący przez obraz. Jeden pomiar opisuje łączny wpływ pikseli leżących na jego drodze. Zbieramy takie pomiary dla wielu promieni i różnych kątów. Ułożone w tablicę tworzą **sinogram** $y$.

W naszym uproszczonym modelu wszystkie pomiary zapisujemy jako $Rx$. Jeden wiersz $R$ odpowiada jednemu pomiarowi, a jedna kolumna — jednemu pikselowi. Macierz $R$ pełni rolę $A$ z sekcji 4. W kodzie każdy piksel dzieli swój wkład między co najwyżej dwie komórki detektora na każdy kąt, więc $R$ jest rzadka.

Dla obrazu $128\times128$ mamy $16\,384$ niewiadome. Przy 90 kątach i 184 komórkach detektora mamy $16\,560$ pomiarów:
$$
R\in\mathbb R^{16560\times16384}.
$$
Pomiarów jest więcej niż niewiadomych, ale nie zapewnia to niezależności kolumn. W tym przykładzie część wierszy jest zerowa, a liczba niezerowych wierszy jest mniejsza od liczby kolumn. Zatem $R$ ma niepełną rangę i $R^\top R$ jest osobliwa. Korzystamy właśnie z uzasadnienia dla macierzy półokreślonej z sekcji 4.2: CG startuje od zera i porusza się w $\operatorname{im}(R^\top)$.

Obraz wzorcowy pobieramy z zestawu testowego `pydicom` i skalujemy do $[0,1]$. Obliczamy pomiary $y_0=Rx_{\mathrm{true}}$ i dodajemy szum gaussowski o odchyleniu standardowym równym $1\%$ największej wartości bezwzględnej $y_0$. Funkcje `A_ct` i `AH_ct` wykonują mnożenia przez $R$ i $R^\top$.

Dla tego modelu funkcja celu i jej gradient są zatem równe
$$
f(x)=\frac12\|Rx-y\|_2^2,\qquad \nabla f(x)=R^\top(Rx-y).
$$
W kodzie gradient obliczylibyśmy jako `AH_ct(A_ct(x) - y)`. Nie tworzymy przy tym macierzy $R^\top R$.

In [ ]:
def load_ct_slice(n=128):
    import pydicom
    from io import BytesIO
    from PIL import Image
    import requests
    url = "https://github.com/pydicom/pydicom/raw/main/src/pydicom/data/test_files/CT_small.dcm"
    ds = pydicom.dcmread(BytesIO(requests.get(url, timeout=30).content))
    img = ds.pixel_array * float(ds.RescaleSlope) + float(ds.RescaleIntercept)  # jednostki Hounsfielda
    img = np.clip(img, -1000, 1000)
    img = (img - img.min()) / (img.max() - img.min())
    img = np.array(Image.fromarray(img).resize((n, n), Image.BILINEAR))
    return img


def ct_matrix(n, angles):
    """Macierz rzutowania R (rzadka): każdy piksel dzieli masę liniowo między 2 komórki detektora."""
    nd = int(np.ceil(n * np.sqrt(2))) + 2
    c = np.arange(n) - (n - 1) / 2
    grid = np.meshgrid(c, c)
    X = grid[0].ravel()
    Y = grid[1].ravel()
    pix = np.arange(n * n)
    rows = []
    cols = []
    vals = []
    for i in range(len(angles)):
        th = angles[i]
        t = X * np.cos(th) + Y * np.sin(th) + (nd - 1) / 2
        j = np.floor(t).astype(int)
        w = t - j
        rows += [i * nd + j, i * nd + j + 1]
        cols += [pix, pix]
        vals += [1 - w, w]
    R = sparse.csr_matrix((np.concatenate(vals),
                           (np.concatenate(rows), np.concatenate(cols))),
                          shape=(len(angles) * nd, n * n))
    return R, nd

In [ ]:
n = 128
x_img = load_ct_slice(n)
x_true = x_img.ravel()

angles = np.deg2rad(np.arange(0, 180, 2))
ct_result = ct_matrix(n, angles)
R = ct_result[0]
nd = ct_result[1]
print(f"R: {R.shape[0]} × {R.shape[1]}, niezerowe: {R.nnz:,} ({100*R.nnz/np.prod(R.shape):.2f}%), "
      f"pamięć ≈ {(R.data.nbytes + R.indices.nbytes + R.indptr.nbytes)/1e6:.1f} MB "
      f"(gęsto: {R.shape[0]*R.shape[1]*8/1e6:.0f} MB)")

rng = np.random.default_rng(0)
y0 = R @ x_true
y = y0 + 0.01 * np.abs(y0).max() * rng.standard_normal(y0.shape)   # zaszumiony sinogram

A_ct = lambda v: R @ v
AH_ct = lambda v: R.T @ v

# test sprzężenia: <A x, z> = <x, Aᴴ z>
xt = rng.normal(size=n*n)
zt = rng.normal(size=R.shape[0])
print("test sprzężenia:", np.vdot(A_ct(xt), zt), np.vdot(xt, AH_ct(zt)))

Sprawdź, jak rośnie zapotrzebowanie na pamięć przy zmianie `n=128` na `n=256`, zachowując tę samą liczbę kątów. Przewidź koszt dla `n=512`. Uruchomienie tego ostatniego wariantu jest opcjonalne: sama macierz rzadka zajmuje około 566 MB, a jej budowa wymaga dodatkowych tablic.

In [ ]:
ks = (2, 5, 20)
n_iter = 60
runs = {m.upper(): normal_eq_solve(A_ct, AH_ct, y, m, n_iter, x_true, save_at=ks) for m in ("gd", "cg")}

show_reconstruction(x_true, y, runs, ks,
                    show_x=lambda v: v.reshape(n, n),
                    show_y=lambda v: v.reshape(len(angles), nd),
                    y_label="sinogram",
                    y_kw=dict(aspect="auto"),
                    res_kw=dict(aspect="auto", vmin=-0.1*y.max(), vmax=0.1*y.max()))
plot_curves(runs, n_iter, "CT: rzadka macierz rzutowania")

### Dlaczego dalsze iteracje mogą pogarszać obraz?

Funkcja celu mierzy zgodność z pomiarami, które zawierają szum, a nie odległość od obrazu wzorcowego. Zjawisko, w którym błąd rekonstrukcji początkowo maleje, a następnie rośnie mimo dalszego zmniejszania residuum danych, nazywamy **semizbieżnością**. Sprawdź na wykresach, czy występuje ono w tym eksperymencie.

1. Po ilu krokach najmniejszy błąd obrazu osiąga CG, a po ilu GD? Porównaj także wartości tych błędów.
2. Czy metoda szybciej zmniejszająca funkcję celu daje też lepszy obraz po tej samej dużej liczbie kroków? Wyjaśnij obserwację i zaproponuj, jak wykorzystać liczbę iteracji do ograniczenia wpływu szumu.
3. Co zmieniłoby usunięcie szumu? Czy niezerowe jądro $R$ pozwala zagwarantować odzyskanie każdego obrazu wzorcowego?
4. W tym eksperymencie znamy $x_{\mathrm{true}}$, więc potrafimy wskazać najlepszą iterację. Dlaczego nie możemy wybierać jej w ten sam sposób przy rzeczywistym pomiarze nieznanego obrazu?

**Wnioski:** …

### 5.2 Rezonans magnetyczny — pomiary z użyciem transformaty Fouriera

W tym przykładzie pomiarami są wybrane współczynniki dwuwymiarowej transformaty Fouriera obrazu. Tablicę tych współczynników nazywamy **k-przestrzenią**. Nie mierzymy jej w całości: maska $M$ ma wartość 1 w miejscach pomiaru i 0 w pozostałych.

Pomiary wykonuje kilka cewek odbiorczych. Każda ma inny rozkład czułości $S_c$, czyli inaczej waży poszczególne piksele. Model pomiarów z cewki $c$ ma postać
$$
y_c=M\,\mathcal F(S_cx),\qquad c=1,\ldots,C.
$$
Mnożenia przez $S_c$ i $M$ wykonujemy punkt po punkcie. $\mathcal F$ jest transformatą Fouriera, obliczaną przez FFT z normalizacją `norm="ortho"`. Operator $A$ zbiera wyniki ze wszystkich cewek.

**Dlaczego teraz liczby zespolone?** Współczynniki Fouriera są na ogół zespolone nawet dla rzeczywistego obrazu. W tym modelu także czułości cewek są zespolone. Nie jest to wymaganie metody CG, lecz cecha modelu pomiarów MRI.

W tym przykładzie dopuszczamy także zespolone $x\in\mathbb C^n$, a na obrazach pokazujemy moduły jego współrzędnych. Iloczyn skalarny ma teraz postać $u^Hv$, gdzie $A^H=\overline A^\top$ oznacza transpozycję ze sprzężeniem. Odpowiednikiem równań normalnych z sekcji 4 jest układ
$$
A^HAx=A^Hy.
$$
Również przy niepełnej randze działa poprzednie uzasadnienie: $\operatorname{im}(A^H)=(\ker A)^\perp$, a $v^HA^HAv=\|Av\|_2^2$. Użyte w kodzie `np.vdot` uwzględnia sprzężenie; kwadraty norm i długość kroku $\alpha_k$ pozostają rzeczywiste.

**Jak obliczamy $A$ i $A^H$ bez tworzenia macierzy?** Funkcja `A_mri` mnoży obraz przez czułości, wykonuje FFT i zachowuje wybrane próbki. Funkcja `AH_mri` wykonuje operacje w odwrotnej kolejności: stosuje maskę, odwrotną FFT, mnoży przez sprzężone czułości i sumuje wyniki cewek:
$$
A^Hy=\sum_{c=1}^C\overline{S_c}\,\mathcal F^{-1}(My_c).
$$
Możemy teraz rozpisać funkcję celu i jej gradient dla wszystkich cewek:
$$
f(x)=\frac12\sum_{c=1}^C\|M\mathcal F(S_cx)-y_c\|_2^2,
$$
$$
\nabla f(x)=A^H(Ax-y)
=\sum_{c=1}^C\overline{S_c}\,\mathcal F^{-1}\!\left(M\left[M\mathcal F(S_cx)-y_c\right]\right).
$$
Gradient względem części rzeczywistych i urojonych $x$ zapisujemy tutaj jako jeden wektor zespolony. Wzór mówi: dla każdej cewki oblicz różnicę między przewidywanymi i zarejestrowanymi pomiarami, zastosuj maskę i odwrotną FFT, pomnóż przez sprzężoną czułość, a na końcu zsumuj wyniki cewek. W kodzie jest to po prostu `AH_mri(A_mri(x) - y)`.

Ponownie $s_k=-\nabla f(x_k)$, a warunek zerowania gradientu daje $A^HAx=A^Hy$. Potrzebny w CG iloczyn obliczamy jako $A^H(Ap)$. Jedna iteracja wymaga $C$ transformat w przód i $C$ odwrotnych. Jawna macierz operatora istnieje, ale jej nie budujemy.

W przygotowanym przykładzie wzorcem jest przekrój szablonu MNI152, utworzonego z uśrednionych skanów MRI. Obraz ma $256\times256$ pikseli, czułości $C=8$ cewek są syntetyczne, a maska zachowuje $1/4$ kolumn k-przestrzeni, w tym jej środek. Mamy zatem $65\,536$ niewiadomych i $131\,072$ zmierzone wartości zespolone. Kod przechowuje również miejsca niezmierzone, wypełnione zerami. Sama liczba pomiarów nie rozstrzyga o randze operatora; różne czułości cewek dostarczają dodatkowej informacji o obrazie.

Uruchom ten sam solver i porównaj trzy wykresy tak jak w CT. Sprawdź, czy w pokazanym zakresie iteracji występuje semizbieżność dla obu metod.

In [ ]:
fft2 = lambda a: np.fft.fft2(a, axes=(-2, -1), norm="ortho")
ifft2 = lambda a: np.fft.ifft2(a, axes=(-2, -1), norm="ortho")


def load_mri_slice(index=110, size=256):
    """Osiowy przekrój szablonu MNI152 (T1, uśrednione MRI 152 osób)."""
    import gzip
    import nibabel as nib
    import requests
    url = "https://github.com/neurolabusc/niivue-images/raw/main/mni152.nii.gz"
    raw = gzip.decompress(requests.get(url, timeout=120).content)
    vol = np.asarray(nib.Nifti1Image.from_bytes(raw).dataobj)
    img = vol[:, :, index].T[::-1]
    out = np.zeros((size, size))
    out[:img.shape[0], (size - img.shape[1])//2:(size - img.shape[1])//2 + img.shape[1]] = img
    return out / out.max()


def coil_maps(ny, nx, n_coils=8):
    """Syntetyczne, gładkie czułości cewek na okręgu; normalizacja sum |S_c|^2 = 1."""
    grid = np.meshgrid(np.linspace(-1, 1, ny), np.linspace(-1, 1, nx), indexing="ij")
    yy = grid[0]
    xx = grid[1]
    S = []
    for c in range(n_coils):
        phi = 2 * np.pi * c / n_coils
        cy = 1.3 * np.sin(phi)
        cx = 1.3 * np.cos(phi)
        mag = np.exp(-((yy - cy)**2 + (xx - cx)**2) / 1.2)
        S.append(mag * np.exp(1j * (0.8 * (xx * np.cos(phi) + yy * np.sin(phi)) + phi)))
    S = np.array(S)
    return S / np.sqrt((np.abs(S)**2).sum(axis=0))


def sampling_mask(ny, nx, accel=4, center=0.08, seed=0):
    """Kartezjańska maska: losowe kolumny (gęściej przy środku) + w pełni próbkowany środek.
    Zwracana w nieprzesuniętym uporządkowaniu FFT."""
    rng = np.random.default_rng(seed)
    f = np.abs(np.linspace(-1, 1, nx))
    prob = (1 - f)**2 + 0.02
    ncenter = int(round(center * nx))
    n_rand = max(nx // accel - ncenter, 0)
    cols = np.zeros(nx, bool)
    mid = nx // 2
    cols[mid - ncenter // 2: mid + (ncenter + 1) // 2] = True
    prob[cols] = 0
    cols[rng.choice(nx, n_rand, replace=False, p=prob / prob.sum())] = True
    M = np.tile(cols, (ny, 1)).astype(float)
    return np.fft.ifftshift(M)

In [ ]:
x_img = load_mri_slice()
ny = x_img.shape[0]
nx = x_img.shape[1]
n_coils = 8
S = coil_maps(ny, nx, n_coils)         # (C, ny, nx)
M = sampling_mask(ny, nx, accel=4)     # (ny, nx), uporządkowanie jak w np.fft.fft2

def A_mri(x):  return M * fft2(S * x)                        # (ny, nx) -> (C, ny, nx)
def AH_mri(y): return (np.conj(S) * ifft2(M * y)).sum(axis=0)

print(f"Próbkowana część k-przestrzeni: {M.mean():.2f}  (przyspieszenie ≈ {1/M.mean():.1f}×)")
print(f"Niewiadome: {ny*nx:,}, dane (próbkowane): {int(n_coils*M.sum()):,}; 2C = {2*n_coils} FFT na iterację")

# test sprzężenia (powinny zgadzać się do ~1e-12)
rng = np.random.default_rng(0)
xt = rng.normal(size=(ny, nx)) + 1j*rng.normal(size=(ny, nx))
zt = rng.normal(size=(n_coils, ny, nx)) + 1j*rng.normal(size=(n_coils, ny, nx))
print("test sprzężenia:", np.vdot(A_mri(xt), zt), np.vdot(xt, AH_mri(zt)))

x_true = x_img.astype(complex)
y0 = A_mri(x_true)
sigma = 0.01 * np.sqrt(np.mean(np.abs(y0[:, M > 0])**2))
y = y0 + M * sigma * (rng.standard_normal(y0.shape) + 1j*rng.standard_normal(y0.shape))  # szum tylko na próbkach

In [ ]:
ks = (2, 5, 15)
n_iter = 40
runs = {m.upper(): normal_eq_solve(A_mri, AH_mri, y, m, n_iter, x_true, save_at=ks) for m in ("gd", "cg")}

ref = np.abs(y[0]).max()
kspace = lambda v: np.log10(np.abs(np.fft.fftshift(v[0])) / ref + 1e-4)   # moduł, skala log, 1. cewka
show_reconstruction(x_true, y, runs, ks,
                    show_x=np.abs, show_y=kspace, y_label="k-przestrzeń, cewka 1, log",
                    y_kw=dict(vmin=-4, vmax=0))
plot_curves(runs, n_iter, "MRI: operator FFT + czułości cewek")
for name in runs:
    print(f"{name}: czas {runs[name]['T'][-1]:.2f} s dla {n_iter} iteracji")

**Pytania do podsumowania:**

1. Co wspólnego mają zadania rekonstrukcji CT i MRI? Jak w każdym z nich obliczamy działanie macierzy równań normalnych bez jej tworzenia?
2. Porównaj liczbę iteracji i koszt uzyskania dobrej rekonstrukcji przez GD i CG. Czy dalsza minimalizacja funkcji celu zawsze poprawia obraz?
3. Czym różni się wpływ szumu od niejednoznaczności wynikającej z niezerowego jądra operatora? Kiedy przy bezszumnych danych możemy zagwarantować odzyskanie obrazu wzorcowego?

**Wnioski:** …

# Zadania

Wykorzystaj kod z zajęć i poprzednich prac domowych. Rozwiązania: `MO_03.ipynb`.

## 1. To samo uwarunkowanie, różne widma — GD i CG

Wróć do zadania 2 z pracy domowej 2 (Lab 2). Dla obu macierzy porównaj GD z CG, zachowując te same dane, punkty startowe, tolerancję i limit iteracji.

Dla każdej macierzy pokaż na wspólnym wykresie względny błąd energetyczny obu metod. Podaj liczbę iteracji, końcową normę residuum i przyczynę zakończenia. Wyjaśnij wpływ rozkładu wartości własnych na wyniki i związek liczby różnych wartości własnych z liczbą kroków CG w arytmetyce dokładnej.

## 2. Nieliniowe CG — Rosenbrock w 40 wymiarach

Korzystając z zadania **D.L.3.2** z kartki `lab_3_D.pdf`, zaimplementuj metody Fletchera–Reevesa (FR) i Polaka–Ribière’a (PR). Przyjmij oznaczenie $\nabla f_k=\nabla f(x_k)$. Gradient w każdym nowym punkcie obliczaj bezpośrednio. **Jeżeli otrzymany kierunek nie jest kierunkiem spadku, zastąp go kierunkiem największego spadku.**

Dokładną minimalizację w kierunku z kartki zastąp **jedną wybraną** procedurą z Lab 1: backtrackingiem albo `golden_fixed`. Użyj tej samej procedury i parametrów z Lab 1 dla GD, FR i PR. Pamiętaj, że dla kierunku $p_k$ pochodna funkcji $\phi(\alpha)=f(x_k+\alpha p_k)$ wynosi $\phi'(0)=(\nabla f_k)^\top p_k$.

Porównaj te trzy metody dla funkcji Rosenbrocka z zadania 3 z pracy domowej 1 (Lab 1), przy $n=40$ i wspólnym starcie $x_0=(-1.2,\ldots,-1.2)$. Przyjmij limit 20 000 iteracji i warunek stopu
$$
\frac{\|\nabla f_k\|_2}{\sqrt{40}}\le10^{-5}.
$$

Na dwóch wspólnych wykresach przedstaw $f(x_k)/39$ oraz $\|\nabla f_k\|_2/\sqrt{40}$. W tabeli podaj liczbę iteracji i wywołań funkcji oraz gradientu podczas optymalizacji, końcową wartość funkcji, normę gradientu i przyczynę zakończenia. Omów skuteczność i koszt metod: czy mniej iteracji oznacza mniej obliczeń? Czy uzyskane wartości funkcji są bliskie minimum globalnemu $0$?

Osiągnięcie limitu iteracji lub niepowodzenie doboru kroku również jest wynikiem eksperymentu — odnotuj je i omów. Nie zakładaj z góry, która metoda okaże się najlepsza.